For this project, the three provided daily files are processed together using wildcard-based batch ingestion. In a production environment, Databricks Auto Loader and checkpoints would be used to process only newly arrived files.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
BASE_PATH = "file:/Workspace/Users/2023pcecashristi050@poornima.org/Drafts"

In [0]:
display(dbutils.fs.ls(BASE_PATH))

path,name,size,modificationTime
file:/Workspace/Users/2023pcecashristi050@poornima.org/Drafts/customers_cdc_2026-04-24.csv,customers_cdc_2026-04-24.csv,15953,1783622505947
file:/Workspace/Users/2023pcecashristi050@poornima.org/Drafts/products_cdc_2026-04-24.csv,products_cdc_2026-04-24.csv,4755,1783622505952
file:/Workspace/Users/2023pcecashristi050@poornima.org/Drafts/orders_incremental_2026-04-24.csv,orders_incremental_2026-04-24.csv,204196,1783622506035
file:/Workspace/Users/2023pcecashristi050@poornima.org/Drafts/Assignment 7 Celebal.ipynb,Assignment 7 Celebal.ipynb,15771,1782930720045
file:/Workspace/Users/2023pcecashristi050@poornima.org/Drafts/orders_incremental_2026-04-25.csv,orders_incremental_2026-04-25.csv,204211,1783622531484
file:/Workspace/Users/2023pcecashristi050@poornima.org/Drafts/products_cdc_2026-04-25.csv,products_cdc_2026-04-25.csv,4895,1783622531527
file:/Workspace/Users/2023pcecashristi050@poornima.org/Drafts/customers_cdc_2026-04-25.csv,customers_cdc_2026-04-25.csv,16020,1783622531575
file:/Workspace/Users/2023pcecashristi050@poornima.org/Drafts/Sample - Superstore.csv,Sample - Superstore.csv,2287806,1782926187864
file:/Workspace/Users/2023pcecashristi050@poornima.org/Drafts/products_batch.csv,products_batch.csv,50712,1783622471066
file:/Workspace/Users/2023pcecashristi050@poornima.org/Drafts/customers_batch.csv,customers_batch.csv,147449,1783622471066


In [0]:
orders_incremental_df = spark.read \
    .option("header", True) \
    .option("inferSchema", False) \
    .csv(f"{BASE_PATH}/orders_incremental_*.csv")

display(orders_incremental_df.limit(5))

order_id,order_ts,customer_id,product_id,store_id,quantity,unit_price,discount_pct,gross_amount,payment_method,order_status,ingest_date,coupon_code
OI3000001,2026-04-26 22:42:00,C01830,P00296,S056,5,64201.56,0.1,256806.24,CARD,returned,2026-04-26,null
OI3000002,2026-04-26 04:25:00,C01705,P00334,S030,2,16032.79,0.0,25652.46,NETBANKING,cancelled,2026-04-26,null
OI3000003,2026-04-23 00:00:00,C01378,P00014,S045,6,24907.63,0.1,141973.49,CARD,shipped,2026-04-26,NEW10
OI3000004,2026-04-26 14:17:00,C02220,P00747,S030,2,61937.51,0.05,117681.27,CARD,shipped,2026-04-26,NEW10
OI3000005,2026-04-26 14:51:00,C01276,P00513,S005,4,72010.17,0.05,273638.65,UPI,delivered,2026-04-26,null


In [0]:
orders_incremental_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_ts: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- discount_pct: string (nullable = true)
 |-- gross_amount: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- ingest_date: string (nullable = true)
 |-- coupon_code: string (nullable = true)



In [0]:
bronze_orders_incremental = (
    orders_incremental_df
    .withColumn("source_file", lit("orders_incremental"))
    .withColumn("ingestion_ts", current_timestamp())
    .withColumn("load_type", lit("incremental"))
)

In [0]:
bronze_orders_incremental.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_raw.bronze_orders_incremental")

In [0]:
spark.sql("""
SELECT COUNT(*) AS total_incremental_orders
FROM retail_raw.bronze_orders_incremental
""").show()

+------------------------+
|total_incremental_orders|
+------------------------+
|                    6135|
+------------------------+



In [0]:
customers_cdc_df = spark.read \
    .option("header", True) \
    .option("inferSchema", False) \
    .csv(f"{BASE_PATH}/customers_cdc_*.csv")

display(customers_cdc_df.limit(5))

customer_id,customer_name,city,segment,gender,signup_date,status,effective_date,operation
C00606,Customer_606,Kolkata,Silver,Other,2025-10-09,active,not_a_date,UPDATE
C00527,Customer_527,Bengaluru,Regular,M,2024-04-10,inactive,2026-04-25,UPDATE
C00660,Customer_660,Jaipur,Regular,F,2024-10-16,active,2026-04-25,UPDATE
C01018,Customer_1018,Jaipur,Platinum,F,2025-04-06,active,2026-04-25,UPDATE
C00781,Customer_781,Pune,Regular,F,2025-12-01,active,2026-04-25,UPDATE


In [0]:
customers_cdc_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- signup_date: string (nullable = true)
 |-- status: string (nullable = true)
 |-- effective_date: string (nullable = true)
 |-- operation: string (nullable = true)



In [0]:
bronze_customers_cdc = (
    customers_cdc_df
    .withColumn("source_file", lit("customers_cdc"))
    .withColumn("ingestion_ts", current_timestamp())
    .withColumn("load_type", lit("incremental"))
)

In [0]:
bronze_customers_cdc.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_raw.bronze_customers_cdc")

In [0]:
spark.sql("""
SELECT COUNT(*) AS total_customers_cdc
FROM retail_raw.bronze_customers_cdc
""").show()

+-------------------+
|total_customers_cdc|
+-------------------+
|                630|
+-------------------+



In [0]:
products_cdc_df = spark.read \
    .option("header", True) \
    .option("inferSchema", False) \
    .csv(f"{BASE_PATH}/products_cdc_*.csv")

display(products_cdc_df.limit(5))

product_id,product_name,category,brand,unit_price,status,created_date,effective_date,operation
P00173,Speaker 173,Electronics,BrandA,50560.38,discontinued,2025-09-11,2026-04-25,UPDATE
P00680,Perfume 680,Beauty,BrandC,40368.75,active,2024-12-24,2026-04-25,UPDATE
P00095,Smartphone 95,Electronics,BrandB,685.97,active,2023-10-15,2026-04-25,UPDATE
P00664,Cream 664,Beauty,BrandB,43522.35,discontinued,2023-03-22,2026-04-25,UPDATE
P00733,Jacket 733,Fashion,BrandA,26764.31,active,2023-07-31,2026-04-25,UPDATE


In [0]:
products_cdc_df.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- status: string (nullable = true)
 |-- created_date: string (nullable = true)
 |-- effective_date: string (nullable = true)
 |-- operation: string (nullable = true)



In [0]:
bronze_products_cdc = (
    products_cdc_df
    .withColumn("source_file", lit("products_cdc"))
    .withColumn("ingestion_ts", current_timestamp())
    .withColumn("load_type", lit("incremental"))
)

In [0]:
bronze_products_cdc.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_raw.bronze_products_cdc")

In [0]:
spark.sql("""
SELECT COUNT(*) AS total_products_cdc
FROM retail_raw.bronze_products_cdc
""").show()

+------------------+
|total_products_cdc|
+------------------+
|               180|
+------------------+



In [0]:
spark.sql("""
SELECT 'bronze_orders_incremental' AS table_name, COUNT(*) AS row_count
FROM retail_raw.bronze_orders_incremental

UNION ALL

SELECT 'bronze_customers_cdc', COUNT(*)
FROM retail_raw.bronze_customers_cdc

UNION ALL

SELECT 'bronze_products_cdc', COUNT(*)
FROM retail_raw.bronze_products_cdc
""").show()

+--------------------+---------+
|          table_name|row_count|
+--------------------+---------+
|bronze_orders_inc...|     6135|
|bronze_customers_cdc|      630|
| bronze_products_cdc|      180|
+--------------------+---------+

